# Chapter 3 — Description Logics
### Notebook 3 · Reasoning services

*Book reference: Section 3.3*

Ontology tools advertise several reasoning services. There is really only one — concept satisfiability — and everything else is a reduction to it. Seeing those reductions is the point of this notebook.

In [ ]:
import sys, os, json, textwrap
from pathlib import Path

# Make the repo root importable no matter where Jupyter was started.
here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "oe_course").is_dir():
        sys.path.insert(0, str(candidate))
        break

import oe_course
print(json.dumps(oe_course.describe_environment(), indent=1))

In [ ]:
import sys, json, logging
from pathlib import Path
here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "oe_course").is_dir():
        sys.path.insert(0, str(candidate)); break
sys.path.insert(0, str(Path.cwd()))

import ch03_toolkit as dl
import pandas as pd
A = dl.Atomic
logging.getLogger("dspy").setLevel(logging.WARNING)

## 1. One service, several names

| Service | Reduces to |
|---|---|
| `C` is satisfiable | *(the primitive)* |
| `C ⊑ D` (subsumption) | `C ⊓ ¬D` is **un**satisfiable |
| `C ≡ D` (equivalence) | `C ⊑ D` and `D ⊑ C` |
| `C`, `D` disjoint | `C ⊓ D` is unsatisfiable |
| KB is consistent | `⊤` is satisfiable wrt the TBox |
| classification | subsumption for every pair |

This is why a DL reasoner is one algorithm and not six.

In [ ]:
tbox = dl.TBox()
tbox.add(A('Dog'), A('Mammal'))
tbox.add(A('Mammal'), A('Animal'))

print('Dog <= Animal :', dl.subsumes(A('Dog'), A('Animal'), tbox))
print('Animal <= Dog :', dl.subsumes(A('Animal'), A('Dog'), tbox))
print()
print('...because Dog and not Animal is',
      'unsatisfiable' if not dl.satisfiable(dl.And(A('Dog'), dl.Not(A('Animal'))),
                                           tbox).satisfiable else 'satisfiable')

## 2. Classification: the service you are really buying

Classification computes the **inferred** hierarchy — subsumptions nobody asserted. This is what makes an ontology more than a data model.

In [ ]:
wildlife = dl.wildlife_tbox()
print('asserted axioms:')
for ax in wildlife.axioms:
    print('  ', ax)

In [ ]:
names = ['Animal', 'Plant', 'Leaf', 'Herbivore', 'Carnivore', 'Giraffe', 'Lion']
inferred = dl.classify(names, wildlife)
asserted = {(dl.to_string(ax.left), dl.to_string(ax.right)) for ax in wildlife.axioms}
print('inferred subsumptions (those NOT asserted verbatim are the payoff):')
for sub, sup in inferred:
    mark = '   ' if (sub, sup) in asserted else ' * '
    print(f'{mark}{sub} <= {sup}')
print('\n* = derived by the reasoner, not written down by anyone.')

## 3. Finding a modelling error the reasoner can see

The classic result from Chapter 1's integration lab, now in DL form. A `Giraffe` is a `Herbivore` (eats only plants) that eats some `Leaf`. A `Carnivore` eats only `Animal`s. `Leaf ⊑ Plant` and `Plant ⊑ ¬Animal`. So:

In [ ]:
both = dl.And(A('Giraffe'), A('Carnivore'))
result = dl.satisfiable(both, wildlife)
print('Giraffe and Carnivore :', result.summary())
assert not result.satisfiable
print('\nThe class is UNSATISFIABLE: nothing can be both. Note that no human\n'
      'wrote a disjointness axiom between Giraffe and Carnivore -- it follows\n'
      'from what they eat. That is the reasoner earning its keep: it found a\n'
      'consequence of axioms written days apart by different people.')

In [ ]:
print('the trace that establishes it:')
for line in dl.satisfiable(both, wildlife, trace=True).trace[:14]:
    print(' ', line)

> **An unsatisfiable class is always a bug.** It means the class can have no instances — so either an axiom is wrong, or the class should never have been created. Protégé shows these in red for good reason. Chapter 5 makes finding them part of a methodology.

### Exercise 3.1 — Derive disjointness you never asserted

Extend the wildlife TBox so that `Lion` and `Herbivore` become provably disjoint **without** writing a disjointness axiom between them. Explain which axioms did the work.

> **Hint.** What stops the Herbivore that a Lion eats from being a Plant?

In [ ]:
# YOUR CODE HERE


<details>
<summary>Solution 3.1</summary>

Run the cell below to check your answer against the reference implementation. The assertions are the grading criteria.

</details>

In [ ]:
t = dl.wildlife_tbox()
print('before:', dl.satisfiable(dl.And(A('Lion'), A('Herbivore')), t).satisfiable)

# A Lion eats some Herbivore; a Herbivore eats only Plants; so if a Lion were a
# Herbivore, the Herbivore it eats would have to be a Plant. Make that impossible:
t.add(A('Herbivore'), A('Animal'))          # already implied, stated for clarity
t.add(A('Animal'), dl.Not(A('Plant')))      # the missing commitment

after = dl.satisfiable(dl.And(A('Lion'), A('Herbivore')), t)
print('after :', after.satisfiable)
assert not after.satisfiable
print('\nThe work was done by Animal <= not Plant. Everything else was already\n'
      'there; the ontology simply had not committed to animals and plants being\n'
      'different kinds of thing. One axiom, and a whole family of errors becomes\n'
      'machine-detectable -- the Chapter 1 no-disjointness smell, from the inside.')

### Exercise 3.2 — Build an unsatisfiable class on purpose

Write a TBox with an unsatisfiable named concept whose unsatisfiability needs at least two axioms to derive, and confirm the reasoner finds it.

In [ ]:
# YOUR CODE HERE


<details>
<summary>Solution 3.2</summary>

Run the cell below to check your answer against the reference implementation. The assertions are the grading criteria.

</details>

In [ ]:
t = dl.TBox()
t.add(A('Teetotaller'), dl.ForAll('drinks', dl.Not(A('Alcohol'))))
t.add(A('WineLover'), dl.Exists('drinks', A('Wine')))
t.add(A('Wine'), A('Alcohol'))
t.add(A('SoberSommelier'), dl.And(A('Teetotaller'), A('WineLover')))

result = dl.satisfiable(A('SoberSommelier'), t)
print('SoberSommelier:', result.summary())
assert not result.satisfiable
# ...and no single axiom is enough on its own:
partial = dl.TBox()
partial.add(A('Teetotaller'), dl.ForAll('drinks', dl.Not(A('Alcohol'))))
partial.add(A('SoberSommelier'), A('Teetotaller'))
assert dl.satisfiable(A('SoberSommelier'), partial).satisfiable
print('\nThree axioms cooperate: the universal restriction, the existential, and\n'
      'the Wine <= Alcohol link. Remove any one and the class becomes satisfiable.\n'
      'This is why unsatisfiability is hard to debug by eye -- the cause is\n'
      'distributed across axioms nobody reads together.')